# Football Match Commentary - Training Pipeline
### Gamma Force | M.Sc. Data Science
**Purpose:** Train all four model phases using pre-extracted feature caches.  
Run `feature_extraction.ipynb` first to populate the cache.

---
**Training phases:**
- Phase 1 - Event detection (TwoStream CNN+LSTM, 7 classes)
- Phase 2 - Tactical GNN (formation classification)
- Phase 3 - Tension regression (MSE on annotation proximity)
- Phase 4 - Language model fine-tuning (GPT-2 commentary head)

**Cells in order:**
1. Spawn guard
2. GPU check
3. Install dependencies
4. Paths - edit here
5. Imports
6. Config - edit hyperparameters
7. Shared utilities
8. Cache-backed dataset classes
9. Model definitions
10. Training helpers & logger
11. Discover matches & caches
12. Phase 1 - Event detection
13. Phase 2 - Tactical GNN
14. Phase 3 - Tension regression
15. Phase 4 - Language model
16. Summary & log tail


## Cell 0 - Spawn Guard
Run **first**. Prevents DataLoader deadlock on Python 3.13.

In [1]:
import multiprocessing
try:
    multiprocessing.set_start_method("spawn", force=True)
    print("OK  Multiprocessing start method: spawn")
except RuntimeError:
    print("OK  Multiprocessing already set:", multiprocessing.get_start_method())


OK  Multiprocessing start method: spawn


## Cell 1 - Check GPU

In [2]:
import torch
print("=" * 55)
print(f"  PyTorch version : {torch.__version__}")
print(f"  CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU             : {torch.cuda.get_device_name(0)}")
    print(f"  VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("  WARNING: No GPU detected")
print("=" * 55)


  PyTorch version : 2.7.1+cu118
  CUDA available  : True
  GPU             : NVIDIA GeForce RTX 2080 Ti
  VRAM            : 11.5 GB


## Cell 2 - Install Dependencies

In [3]:
%pip install -q av transformers tqdm opencv-python pandas

import torch
TORCH_VER = torch.__version__.split("+")[0]
CUDA_VER  = "cu" + torch.version.cuda.replace(".", "") if torch.cuda.is_available() else "cpu"
print(f"Installing torch-geometric for torch={TORCH_VER}, cuda={CUDA_VER}")
%pip install -q torch-geometric

print("\nOK All dependencies installed.")


Note: you may need to restart the kernel to use updated packages.
Installing torch-geometric for torch=2.7.1, cuda=cu118
Note: you may need to restart the kernel to use updated packages.

OK All dependencies installed.


## Cell 3 - Paths & Run Settings
> Edit `BASE` to match `feature_extraction.ipynb`.  
> Edit `RUN_ID` and `TOTAL_RUNS` to control which chunk trains this session.

**How partial training works:**
- All your cached matches are split into `TOTAL_RUNS` equal chunks
- Each session you set `RUN_ID = 1`, `2`, `3`, or `4` and run the notebook
- Each run resumes Phase 1/3 from the last saved checkpoint automatically
- After all runs complete, every match will have contributed to training

**Example schedule:**
- Session 1: `RUN_ID=1` — trains on matches 1-25%
- Session 2: `RUN_ID=2` — resumes checkpoint, trains on matches 26-50%
- Session 3: `RUN_ID=3` — resumes checkpoint, trains on matches 51-75%
- Session 4: `RUN_ID=4` — resumes checkpoint, trains on matches 76-100%


In [4]:
import os
from pathlib import Path

# ── Edit THESE ───────────────────────────────────────────────────────────────
BASE = Path.home() / "Desktop" / "Gamma Force"
# Colab: BASE = Path("/content/drive/MyDrive/DL_Project")

# Partial training settings
# Set RUN_ID to 1, 2, 3, or 4 before each session.
# Set TOTAL_RUNS to however many sessions you want (2, 3, or 4).
RUN_ID     = 4   # <-- CHANGE THIS each session (1, 2, 3, 4)
TOTAL_RUNS = 4   # <-- Total number of sessions you plan to run
# ─────────────────────────────────────────────────────────────────────────────

DATASET_DIR       = str(BASE / "matches")
CACHE_DIR         = str(BASE / "feature_cache")
EVENT_CACHE_DIR   = str(BASE / "feature_cache" / "event_clips")
TENSION_CACHE_DIR = str(BASE / "feature_cache" / "tension_clips")
FULL_FEAT_DIR     = str(BASE / "feature_cache" / "full_features")
CKPT_DIR          = str(BASE / "checkpoints")
LOG_FILE          = str(BASE / "training_log.csv")

Path(CKPT_DIR).mkdir(parents=True, exist_ok=True)

print(f"Base dir          : {BASE}")
print(f"Event cache       : {EVENT_CACHE_DIR}")
print(f"Tension cache     : {TENSION_CACHE_DIR}")
print(f"Checkpoints       : {CKPT_DIR}")
print(f"Training log      : {LOG_FILE}")
print(f"\nRUN_ID            : {RUN_ID} / {TOTAL_RUNS}")

assert 1 <= RUN_ID <= TOTAL_RUNS, \
    f"RUN_ID must be between 1 and TOTAL_RUNS ({TOTAL_RUNS}), got {RUN_ID}"

for label, path in [("Event cache", EVENT_CACHE_DIR),
                     ("Tension cache", TENSION_CACHE_DIR)]:
    exists  = Path(path).exists()
    n_files = len(list(Path(path).rglob("*.pt"))) if exists else 0
    status  = f"OK ({n_files} .pt files)" if n_files > 0 else "WARNING: empty or missing"
    print(f"  {label:<20}: {status}")


Base dir          : /home/sysadm/Desktop/Gamma Force
Event cache       : /home/sysadm/Desktop/Gamma Force/feature_cache/event_clips
Tension cache     : /home/sysadm/Desktop/Gamma Force/feature_cache/tension_clips
Checkpoints       : /home/sysadm/Desktop/Gamma Force/checkpoints
Training log      : /home/sysadm/Desktop/Gamma Force/training_log.csv

RUN_ID            : 4 / 4
  Event cache         : OK (200 .pt files)
  Tension cache       : OK (200 .pt files)


## Cell 4 - Imports

In [5]:
import os, time, csv, json, logging
from pathlib import Path
from dataclasses import dataclass
from typing import List, Tuple, Optional
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.amp import GradScaler, autocast
import torchvision.models as models
import numpy as np
from tqdm.auto import tqdm
import pandas as pd

try:
    import av
    print("OK PyAV loaded")
except ImportError:
    av = None
    print("WARNING: PyAV not found")

try:
    from transformers import GPT2LMHeadModel, GPT2Tokenizer
    print("OK Transformers loaded")
except ImportError:
    GPT2LMHeadModel = GPT2Tokenizer = None
    print("WARNING: transformers not found - Phase 4 will be skipped")

try:
    from torch_geometric.nn import GCNConv
    HAS_GEO = True
    print("OK torch-geometric loaded")
except ImportError:
    HAS_GEO = False
    print("WARNING: torch-geometric not found - GNN will use linear fallback")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)
log = logging.getLogger("GammaForce.Train")
print("\nOK All imports done.")


OK PyAV loaded
OK Transformers loaded
OK torch-geometric loaded

OK All imports done.


## Cell 5 - Config
> Edit hyperparameters here.  
> Keep `sample_fps`, `img_size`, `clip_before_sec`, `clip_after_sec`, `cnn_backbone`, `feature_dim` **identical** to the extraction notebook.

In [6]:
@dataclass
class TrainConfig:
    dataset_dir:       str   = DATASET_DIR
    event_cache_dir:   str   = EVENT_CACHE_DIR
    tension_cache_dir: str   = TENSION_CACHE_DIR
    full_feat_dir:     str   = FULL_FEAT_DIR
    ckpt_dir:          str   = CKPT_DIR
    log_file:          str   = LOG_FILE

    # Must match extraction notebook exactly
    sample_fps:       float = 5.0
    img_size:         int   = 224
    clip_before_sec:  float = 4.0
    clip_after_sec:   float = 6.0
    cnn_backbone:     str   = "resnet18"
    feature_dim:      int   = 512
    use_optical_flow: bool  = True

    # Event detection
    temporal_model:    str   = "two_stream"
    num_event_classes: int   = 7
    event_epochs:      int   = 40
    event_lr:          float = 5e-5

    # Tactical GNN
    num_players:     int   = 22
    gnn_hidden:      int   = 128
    num_formations:  int   = 10
    tactical_epochs: int   = 15
    tactical_lr:     float = 1e-3

    # Tension regression
    tension_epochs: int   = 10
    tension_lr:     float = 5e-4

    # Language model
    lm_model_name:      str   = "gpt2"
    lm_epochs:          int   = 5
    lm_lr:              float = 5e-5
    max_commentary_len: int   = 128

    # Training
    batch_size:       int   = 16
    grad_accum_steps: int   = 8
    num_workers:      int   = 0
    mixed_precision:  bool  = True
    pin_memory:       bool  = False
    seed:             int   = 42
    val_split:        float = 0.15

    # Derived
    clip_len: int = 50


cfg = TrainConfig()
cfg.clip_len = int((cfg.clip_before_sec + cfg.clip_after_sec) * cfg.sample_fps)

torch.manual_seed(cfg.seed)
np.random.seed(cfg.seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    torch.backends.cudnn.benchmark = True
    cfg.mixed_precision = True
    cfg.pin_memory      = False
    cfg.num_workers     = 0
else:
    cfg.mixed_precision = False
    cfg.pin_memory      = False

print(f"Device          : {device}")
if device.type == "cuda":
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
print(f"Clip length     : {cfg.clip_len} frames  ({cfg.clip_before_sec}+{cfg.clip_after_sec}s @ {cfg.sample_fps}fps)")
print(f"Mixed precision : {cfg.mixed_precision}")
print(f"Batch size      : {cfg.batch_size}  (accum {cfg.grad_accum_steps} steps = effective {cfg.batch_size*cfg.grad_accum_steps})")


Device          : cuda
GPU             : NVIDIA GeForce RTX 2080 Ti
Clip length     : 50 frames  (4.0+6.0s @ 5.0fps)
Mixed precision : True
Batch size      : 16  (accum 8 steps = effective 128)


## Cell 6 - Shared Utilities
Annotation constants, class weights, discover helpers.

In [7]:
SOCCERNET_TO_IDX = {
    "goal": 2, "shots on target": 0, "shots off target": 0,
    "corner": 3, "foul": 1, "offside": 4,
    "direct free-kick": 1, "indirect free-kick": 1,
    "yellow card": 1, "red card": 1, "kick-off": 5,
    "throw-in": 6, "ball out of play": 6, "clearance": 6, "substitution": 6,
}
EVENT_NAMES   = ["Shot", "Foul", "Goal", "Corner", "Offside", "Kickoff", "Other"]
EVENT_TENSION = {2: 1.0, 0: 0.85, 1: 0.6, 3: 0.5, 4: 0.4, 5: 0.3, 6: 0.1}
NUM_CLASSES   = 7
VIDEO_EXTENSIONS = (".mp4", ".mkv", ".avi", ".mov", ".m4v", ".webm")


def find_video_halves(match_dir):
    halves = []
    for ext in VIDEO_EXTENSIONS:
        halves.extend(match_dir.glob(f"*{ext}"))
    return sorted(halves)


def discover_matches(dataset_dir):
    root = Path(dataset_dir)
    if not root.exists():
        log.error(f"Dataset dir not found: {dataset_dir}")
        return []
    matches = []
    for match_dir in sorted(root.iterdir()):
        if not match_dir.is_dir(): continue
        halves = find_video_halves(match_dir)
        if len(halves) < 2:
            log.warning(f"[SKIP] {match_dir.name}")
            continue
        matches.append((halves[0], halves[1]))
    log.info(f"Discovered {len(matches)} matches")
    return matches


def compute_class_weights(labels, num_classes, device):
    counts  = Counter(labels)
    total   = sum(counts.values())
    weights = torch.tensor(
        [total / counts.get(i, 1) for i in range(num_classes)], dtype=torch.float)
    weights = torch.sqrt(weights) / torch.sqrt(weights).median()
    return weights.to(device)


def save_checkpoint(model, name, epoch, ckpt_dir):
    Path(ckpt_dir).mkdir(parents=True, exist_ok=True)
    path = Path(ckpt_dir) / f"{name}_epoch{epoch:03d}.pt"
    torch.save(model.state_dict(), path)
    log.info(f"  Saved: {path.name}")


def load_best_checkpoint(model, name, ckpt_dir, device):
    ckpts = sorted(Path(ckpt_dir).glob(f"{name}_*.pt"))
    if not ckpts:
        print(f"  No checkpoint found for {name} - using untrained model")
        return model
    best = ckpts[-1]
    try:
        state = torch.load(best, map_location=device, weights_only=True)
        model.load_state_dict(state, strict=True)
        print(f"  OK Loaded: {best.name}")
        return model
    except RuntimeError as e:
        if "size mismatch" in str(e):
            print(f"  WARNING: {best.name} is stale - deleting.")
            for ck in Path(ckpt_dir).glob(f"{name}_*.pt"):
                ck.unlink()
        else:
            print(f"  ERROR loading {best.name}: {e}")
        return model


def build_edge_index(num_players):
    src  = torch.arange(num_players).repeat_interleave(num_players)
    dst  = torch.arange(num_players).repeat(num_players)
    mask = src != dst
    return torch.stack([src[mask], dst[mask]])


print(f"OK Shared utilities ready.")
print(f"   Classes ({NUM_CLASSES}): {EVENT_NAMES}")


def resume_checkpoint(model, optimizer, scheduler, name, ckpt_dir, device):
    # Load latest checkpoint for incremental training.
    # Returns (model, optimizer, scheduler, start_epoch)
    # start_epoch is the epoch AFTER the last saved one.
    ckpts = sorted(Path(ckpt_dir).glob(f"{name}_epoch*.pt"))
    if not ckpts:
        print(f"  No checkpoint for {name} - starting from epoch 1")
        return model, optimizer, scheduler, 1
    latest = ckpts[-1]
    # Parse epoch number from filename e.g. event_detector_epoch012.pt
    try:
        last_epoch = int(latest.stem.split("epoch")[-1])
    except ValueError:
        last_epoch = 0
    try:
        state = torch.load(latest, map_location=device, weights_only=True)
        model.load_state_dict(state, strict=True)
        # Fast-forward scheduler to match saved epoch
        for _ in range(last_epoch):
            scheduler.step()
        print(f"  Resumed: {latest.name}  (continuing from epoch {last_epoch + 1})")
        return model, optimizer, scheduler, last_epoch + 1
    except Exception as e:
        print(f"  WARNING: Could not resume {latest.name}: {e}")
        print(f"  Starting from epoch 1.")
        return model, optimizer, scheduler, 1


def slice_matches(all_matches, run_id, total_runs):
    # Split all_matches into total_runs equal chunks and return chunk run_id.
    n      = len(all_matches)
    size   = max(1, (n + total_runs - 1) // total_runs)  # ceiling division
    start  = (run_id - 1) * size
    end    = min(run_id * size, n)
    chunk  = all_matches[start:end]
    print(f"  Run {run_id}/{total_runs}: matches [{start+1}..{end}] "
          f"({len(chunk)} of {n} total)")
    return chunk


OK Shared utilities ready.
   Classes (7): ['Shot', 'Foul', 'Goal', 'Corner', 'Offside', 'Kickoff', 'Other']


## Cell 7 - Cache-Backed Dataset Classes
These datasets load directly from the `.pt` files written by `feature_extraction.ipynb`.  
**No video decoding happens here** - training is fast.

In [8]:
class CachedEventDataset(Dataset):
    # Loads from feature_cache/event_clips/<match>/<stem>_events.pt
    # Each .pt file is a list of dicts: {feat, flow, label}
    def __init__(self, matches, cfg):
        # matches is now a list of match_name strings (not video path tuples)
        self.samples  = []
        self.use_flow = cfg.use_optical_flow

        print("  Loading CachedEventDataset...")
        missing = 0
        for match_name in tqdm(matches, desc="  Matches", leave=False):
            match_cache_dir = Path(cfg.event_cache_dir) / match_name
            for cache_file in sorted(match_cache_dir.glob("*_events.pt")):
                if not cache_file.exists():
                    log.warning(f"  Missing cache: {cache_file}")
                    missing += 1
                    continue
                try:
                    data = torch.load(cache_file, map_location="cpu",
                                      weights_only=False)
                    for sample in data:
                        self.samples.append(
                            (sample["feat"], sample["flow"], sample["label"]))
                except Exception as e:
                    log.warning(f"  Failed to load {cache_file.name}: {e}")

        if missing:
            print(f"  WARNING: {missing} cache files missing - "
                  "run feature_extraction.ipynb first.")

        dist = Counter(label for _, _, label in self.samples)
        print(f"  CachedEventDataset: {len(self.samples)} clips")
        for i, name in enumerate(EVENT_NAMES):
            print(f"    {name:<10}: {dist.get(i, 0)}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        feat, flow, label = self.samples[idx]
        return feat, flow, torch.tensor(label, dtype=torch.long)


class CachedTensionDataset(Dataset):
    # Loads from feature_cache/tension_clips/<match>/<stem>_tension.pt
    # Each .pt file is a list of dicts: {feat, tension}
    def __init__(self, matches, cfg):
        # matches is now a list of match_name strings (not video path tuples)
        self.samples = []
        print("  Loading CachedTensionDataset...")
        missing = 0
        for match_name in tqdm(matches, desc="  Matches", leave=False):
            match_cache_dir = Path(cfg.tension_cache_dir) / match_name
            for cache_file in sorted(match_cache_dir.glob("*_tension.pt")):
                if not cache_file.exists():
                    missing += 1
                    continue
                try:
                    data = torch.load(cache_file, map_location="cpu",
                                      weights_only=False)
                    for sample in data:
                        self.samples.append((sample["feat"], sample["tension"]))
                except Exception as e:
                    log.warning(f"  Failed to load {cache_file.name}: {e}")

        if missing:
            print(f"  WARNING: {missing} cache files missing.")

        avg = sum(t for _, t in self.samples) / max(len(self.samples), 1)
        print(f"  CachedTensionDataset: {len(self.samples)} clips  "
              f"(avg tension: {avg:.3f})")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        feat, tension = self.samples[idx]
        return feat, torch.tensor(tension, dtype=torch.float32)


class _SyntheticTrajectoryDataset(Dataset):
    # Fallback when no real tracking data is available
    def __init__(self, size=1000, num_players=22):
        self.size = size
        self.num_players = num_players
        self.edge_index  = build_edge_index(num_players)
    def __len__(self): return self.size
    def __getitem__(self, idx):
        nf = torch.randn(self.num_players, 4)
        return nf, self.edge_index, torch.randint(0, 10, (1,)).squeeze()


class TrajectoryDataset(Dataset):
    def __init__(self, tracking_dir, split="train", num_players=22, stride=5):
        self.samples     = []
        self.num_players = num_players
        self.edge_index  = build_edge_index(num_players)
        gt_files = sorted(Path(tracking_dir).glob(f"{split}/*/gt/gt.txt"))
        if not gt_files:
            raise FileNotFoundError(
                f"No gt.txt under {tracking_dir}/{split}/*/gt/")
        print(f"  Loading {len(gt_files)} tracking clips [{split}]...")
        for gt_path in tqdm(gt_files, desc="  Tracking", leave=False):
            try:
                frames = self._load_clip(gt_path, num_players)
            except Exception as e:
                log.warning(f"Skip: {e}"); continue
            for fid in sorted(frames.keys()):
                if fid % stride != 0: continue
                nf = frames[fid]
                self.samples.append((nf, self._formation_label(nf)))
        print(f"  OK {len(self.samples)} tracking samples.")

    @staticmethod
    def _load_clip(gt_path, max_players=22, img_w=1920, img_h=1080):
        PLAYER_CLASSES = {1, 2, 3, 4}
        df = pd.read_csv(str(gt_path), header=None,
                         names=["frame_id","track_id","left","top",
                                "width","height","conf","class",
                                "visibility","unused"])
        df = df[df["class"].isin(PLAYER_CLASSES)].copy()
        df["cx"] = (df["left"] + df["width"]  / 2) / img_w
        df["cy"] = (df["top"]  + df["height"] / 2) / img_h
        df["nw"] = df["width"]  / img_w
        df["nh"] = df["height"] / img_h
        frames = {}
        for fid, group in df.groupby("frame_id"):
            coords = group[["cx","cy","nw","nh"]].values.astype(np.float32)
            if len(coords) < max_players:
                pad    = np.zeros((max_players - len(coords), 4), dtype=np.float32)
                coords = np.vstack([coords, pad])
            else:
                coords = coords[:max_players]
            frames[int(fid)] = torch.tensor(coords)
        return frames

    @staticmethod
    def _formation_label(nf):
        valid = nf[nf[:, 0] > 0]
        if len(valid) < 4: return 0
        sx = valid[:, 0].sort().values; n = len(sx)
        return min(int((sx[2*n//3:].mean()-sx[:n//3].mean()).item()*10), 9)

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        nf, label = self.samples[idx]
        return nf, self.edge_index, torch.tensor(label, dtype=torch.long)


print("OK All dataset classes defined.")


OK All dataset classes defined.


## Cell 8 - Model Definitions

In [9]:
def build_backbone(name, feature_dim):
    if name == "resnet18":
        m = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        m.fc = nn.Linear(512, feature_dim)
    elif name == "resnet50":
        m = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        m.fc = nn.Linear(2048, feature_dim)
    elif name == "efficientnet_b0":
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, feature_dim)
    else:
        raise ValueError(f"Unknown backbone: {name}")
    return m


class CNNLSTMEventDetector(nn.Module):
    # Bidirectional LSTM on appearance features only (fallback)
    def __init__(self, feat_dim, num_classes, hidden=256):
        super().__init__()
        self.lstm = nn.LSTM(feat_dim, hidden, num_layers=2,
                            batch_first=True, dropout=0.2, bidirectional=True)
        self.head = nn.Linear(hidden * 2, num_classes)
    def forward(self, x, flow=None):
        out, _ = self.lstm(x)
        return self.head(out[:, -1])


class TwoStreamEventDetector(nn.Module):
    # Two-stream event detector.
    # Stream 1 (Appearance): CNN features through BiLSTM.
    # Stream 2 (Motion): Optical flow through Conv+LSTM.
    # Falls back to appearance-only when flow is None.
    def __init__(self, feat_dim, num_classes, hidden=256, img_size=224):
        super().__init__()
        self.appear_lstm = nn.LSTM(feat_dim, hidden, num_layers=2,
                                   batch_first=True, dropout=0.2, bidirectional=True)
        self.appear_norm = nn.LayerNorm(hidden * 2)
        self.motion_encoder = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=5, stride=2, padding=2),
            nn.ReLU(),
            nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(4),
            nn.Flatten(),
            nn.Linear(32 * 4 * 4, hidden // 2),
            nn.ReLU(),
        )
        self.motion_lstm = nn.LSTM(hidden // 2, hidden // 2, num_layers=1,
                                   batch_first=True, bidirectional=True)
        self.motion_norm = nn.LayerNorm(hidden)
        self.fusion = nn.Sequential(
            nn.Linear(hidden * 2 + hidden, hidden),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden, num_classes),
        )

    def forward(self, appear_feat, flow_feat=None):
        # appear_feat: (B, T, feat_dim)
        # flow_feat:   (B, T-1, 1, H, W) or None
        app_out, _ = self.appear_lstm(appear_feat)
        app_vec    = self.appear_norm(app_out[:, -1])
        if flow_feat is not None and flow_feat.shape[1] > 0:
            B, Tf, C, H, W = flow_feat.shape
            mot_enc = self.motion_encoder(flow_feat.view(B * Tf, C, H, W))
            mot_enc = mot_enc.view(B, Tf, -1)
            mot_out, _ = self.motion_lstm(mot_enc)
            mot_vec    = self.motion_norm(mot_out[:, -1])
        else:
            mot_vec = torch.zeros(
                appear_feat.shape[0],
                self.motion_norm.normalized_shape[0],
                device=appear_feat.device)
        fused = torch.cat([app_vec, mot_vec], dim=-1)
        return self.fusion(fused)


def build_temporal_model(cfg):
    if cfg.temporal_model == "two_stream":
        return TwoStreamEventDetector(
            cfg.feature_dim, cfg.num_event_classes, img_size=cfg.img_size)
    elif cfg.temporal_model == "cnn_lstm":
        return CNNLSTMEventDetector(cfg.feature_dim, cfg.num_event_classes)
    raise ValueError(f"Unknown temporal_model: {cfg.temporal_model}")


class TacticalGNN(nn.Module):
    def __init__(self, in_feat, hidden, num_classes):
        super().__init__()
        self.use_geo = HAS_GEO
        if HAS_GEO:
            self.conv1 = GCNConv(in_feat, hidden)
            self.conv2 = GCNConv(hidden, hidden)
        else:
            self.conv1 = nn.Linear(in_feat, hidden)
            self.conv2 = nn.Linear(hidden, hidden)
        self.head = nn.Linear(hidden, num_classes)
        self.relu = nn.ReLU()

    def forward(self, node_feats, edge_index):
        x = self.relu(self.conv1(node_feats, edge_index)
                      if self.use_geo else self.conv1(node_feats))
        x = self.relu(self.conv2(x, edge_index)
                      if self.use_geo else self.conv2(x))
        return self.head(x.mean(dim=-2))


class TensionRegressor(nn.Module):
    def __init__(self, feat_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(feat_dim, 256), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(256, 64),       nn.ReLU(),
            nn.Linear(64, 1),         nn.Sigmoid(),
        )
    def forward(self, x):
        return self.net(x.mean(dim=1)).squeeze(-1)


class CommentaryHead(nn.Module):
    def __init__(self, state_dim, lm_embed_dim):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(state_dim, 256), nn.GELU(),
            nn.Linear(256, lm_embed_dim),
        )
    def forward(self, state):
        return self.proj(state)


print("OK All model classes defined.")
print("   TwoStreamEventDetector: appearance (BiLSTM) + motion (Conv+LSTM)")
print("   TacticalGNN, TensionRegressor, CommentaryHead")


OK All model classes defined.
   TwoStreamEventDetector: appearance (BiLSTM) + motion (Conv+LSTM)
   TacticalGNN, TensionRegressor, CommentaryHead


## Cell 9 - Training Helpers & Logger

In [10]:
def make_loader(dataset, cfg, shuffle=True):
    return DataLoader(
        dataset,
        batch_size=cfg.batch_size,
        shuffle=shuffle,
        num_workers=cfg.num_workers,
        persistent_workers=(cfg.num_workers > 0),
        pin_memory=cfg.pin_memory and cfg.num_workers > 0,
        drop_last=True,
    )


def train_one_epoch_event(model, loader, optimizer, criterion, scaler, cfg, device):
    # Training loop for CachedEventDataset: batches are (feat, flow, label)
    model.train()
    total_loss = 0.0
    optimizer.zero_grad()
    pbar = tqdm(enumerate(loader), total=len(loader), leave=False, desc="  batches")
    for step, (feat, flow, y) in pbar:
        feat = feat.to(device, non_blocking=True)
        flow = flow.to(device, non_blocking=True) if cfg.use_optical_flow else None
        y    = y.to(device, non_blocking=True)
        with autocast(device_type=device.type, enabled=cfg.mixed_precision):
            loss = criterion(model(feat, flow), y) / cfg.grad_accum_steps
        scaler.scale(loss).backward()
        if (step + 1) % cfg.grad_accum_steps == 0:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
        total_loss += loss.item() * cfg.grad_accum_steps
        pbar.set_postfix(loss=f"{loss.item()*cfg.grad_accum_steps:.4f}")
    return total_loss / max(len(loader), 1)


def train_one_epoch(model, loader, optimizer, criterion, scaler, cfg, device):
    # Standard training loop for (x, y) datasets: tension and tactical
    model.train()
    total_loss = 0.0
    optimizer.zero_grad()
    pbar = tqdm(enumerate(loader), total=len(loader), leave=False, desc="  batches")
    for step, (x, y) in pbar:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        with autocast(device_type=device.type, enabled=cfg.mixed_precision):
            loss = criterion(model(x), y) / cfg.grad_accum_steps
        scaler.scale(loss).backward()
        if (step + 1) % cfg.grad_accum_steps == 0:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
        total_loss += loss.item() * cfg.grad_accum_steps
        pbar.set_postfix(loss=f"{loss.item()*cfg.grad_accum_steps:.4f}")
    return total_loss / max(len(loader), 1)


class TrainingLogger:
    # Writes one row per epoch to training_log.csv.
    # Persists across Colab sessions - every run appends new rows.
    COLUMNS = ["timestamp","phase","epoch","total_epochs",
               "train_loss","val_loss","best_val_loss",
               "epoch_time_sec","matches_trained","notes"]

    def __init__(self, log_file, n_matches=0):
        self.log_file  = log_file
        self.n_matches = n_matches
        existed = Path(log_file).exists()
        if not existed:
            with open(log_file, "w", newline="") as f:
                csv.DictWriter(f, fieldnames=self.COLUMNS).writeheader()
            print(f"  Created training log : {log_file}")
        else:
            n = len(pd.read_csv(log_file))
            print(f"  Appending to existing log ({n} rows) : {log_file}")

    def log(self, phase, epoch, total_epochs, train_loss,
            val_loss=None, best_val_loss=None, epoch_time=0.0, notes=""):
        row = {
            "timestamp":       time.strftime("%Y-%m-%d %H:%M:%S"),
            "phase":           phase,
            "epoch":           epoch,
            "total_epochs":    total_epochs,
            "train_loss":      f"{train_loss:.6f}",
            "val_loss":        f"{val_loss:.6f}"      if val_loss      is not None else "",
            "best_val_loss":   f"{best_val_loss:.6f}" if best_val_loss is not None else "",
            "epoch_time_sec":  f"{epoch_time:.1f}",
            "matches_trained": self.n_matches,
            "notes":           notes,
        }
        with open(self.log_file, "a", newline="") as f:
            csv.DictWriter(f, fieldnames=self.COLUMNS).writerow(row)

    def summary(self):
        if not Path(self.log_file).exists():
            print("No training log found."); return
        df = pd.read_csv(self.log_file)
        print(f"\n{'='*65}")
        print(f"  Training Log Summary  -  {len(df)} epochs recorded")
        print(f"{'='*65}")
        for phase, grp in df.groupby("phase", sort=False):
            first = float(grp["train_loss"].iloc[0])
            best  = float(grp["train_loss"].min())
            last  = float(grp["train_loss"].iloc[-1])
            secs  = grp["epoch_time_sec"].astype(float).sum()
            print(f"  {phase:<25}  {len(grp):>3} epochs  "
                  f"start={first:.4f}  best={best:.4f}  last={last:.4f}  "
                  f"time={secs/60:.1f}min")
        print(f"{'='*65}")

    def tail(self, n=20):
        if not Path(self.log_file).exists():
            print("No training log found."); return
        df = pd.read_csv(self.log_file)
        print(f"\nLast {min(n, len(df))} logged epochs:")
        print(df.tail(n).to_string(index=False))


print("OK Training helpers and TrainingLogger defined.")


OK Training helpers and TrainingLogger defined.


## Cell 10 - Discover Matches & Slice for This Run
Finds all cached matches, then selects only the chunk for `RUN_ID`.

In [11]:
def discover_matches_from_cache(cfg):
    # Discovers matches from feature cache folders.
    # Works even when original videos have been deleted.
    event_root = Path(cfg.event_cache_dir)
    if not event_root.exists():
        log.error(f"Event cache dir not found: {event_root}")
        return []
    match_names = sorted([d.name for d in event_root.iterdir() if d.is_dir()])
    if not match_names:
        log.error(f"No match folders found in cache: {event_root}")
        return []
    all_matches = []
    for match_name in match_names:
        event_files = sorted((event_root / match_name).glob("*_events.pt"))
        if len(event_files) < 2:
            log.warning(f"[SKIP] {match_name} - only {len(event_files)} half(s) cached")
            continue
        all_matches.append(match_name)
    log.info(f"Total cached matches: {len(all_matches)}")
    return all_matches


# Discover ALL matches from cache
all_matches = discover_matches_from_cache(cfg)

if not all_matches:
    raise RuntimeError(
        f"No cached matches found in '{cfg.event_cache_dir}'.\n"
        "Run feature_extraction.ipynb first to populate the cache.")

# Slice to only this run's chunk
matches = slice_matches(all_matches, RUN_ID, TOTAL_RUNS)

if not matches:
    raise RuntimeError(f"No matches in chunk for RUN_ID={RUN_ID}. "
                       f"Check TOTAL_RUNS vs actual match count ({len(all_matches)}).")

print(f"\n{'='*55}")
print(f"  Total cached matches : {len(all_matches)}")
print(f"  This run (RUN_ID={RUN_ID})  : {len(matches)} matches")
print(f"{'='*55}")
for m in matches:
    print(f"  {m}")

training_logger = TrainingLogger(cfg.log_file, n_matches=len(matches))


15:35:37  INFO      Total cached matches: 100


  Run 4/4: matches [76..100] (25 of 100 total)

  Total cached matches : 100
  This run (RUN_ID=4)  : 25 matches
  2017-02-04 - 20-30 Dortmund 1 - 0 RB Leipzig
  2017-02-11 - 17-30 Darmstadt 2 - 1 Dortmund
  2017-02-14 - 22-45 Paris SG 4 - 0 Barcelona
  2017-02-15 - 22-45 Real Madrid 3 - 1 Napoli
  2017-02-26 - 18-15 Atl. Madrid 1 - 2 Barcelona
  2017-02-27 - 23-00 Leicester 3 - 1 Liverpool
  2017-03-04 - 22-45 Barcelona 5 - 0 Celta Vigo
  2017-03-07 - 22-45 Napoli 1 - 3 Real Madrid
  2017-03-12 - 23-00 Lorient 1 - 2 Paris SG
  2017-03-17 - 22-30 Dortmund 1 - 0 Ingolstadt
  2017-03-18 - 18-15 Ath Bilbao 1 - 2 Real Madrid
  2017-03-19 - 22-45 Barcelona 4 - 2 Valencia
  2017-03-19 - 23-00 Paris SG 2 - 1 Lyon
  2017-04-01 - 16-30 Schalke 1 - 1 Dortmund
  2017-04-09 - 18-00 Everton 4 - 2 Leicester
  2017-04-15 - 16-30 Dortmund 3 - 1 Eintracht Frankfurt
  2017-04-18 - 19-30 Metz 2 - 3 Paris SG
  2017-04-18 - 21-45 Real Madrid 4 - 2 Bayern Munich
  2017-04-23 - 21-45 Real Madrid 2 - 3 Barcel

## Cell 11 - Phase 1: Event Detection
Trains on this run's match chunk. **Automatically resumes from the last saved checkpoint** so each run continues where the previous one left off.

In [12]:
def phase1_event_detection(matches, cfg, device):
    print("=" * 55)
    print(f"  PHASE 1 - Event Detection  [Run {RUN_ID}/{TOTAL_RUNS}]")
    print("=" * 55)

    dataset = CachedEventDataset(matches, cfg)

    if len(dataset) == 0:
        raise RuntimeError(
            "CachedEventDataset is empty.\n"
            "Run feature_extraction.ipynb first to populate the cache.")

    if len(dataset) < cfg.batch_size * 2:
        raise RuntimeError(
            f"Too few clips ({len(dataset)}) for batch_size={cfg.batch_size}.\n"
            "Try reducing batch_size in Config or add more matches to this run.")

    n_val = max(1, int(len(dataset) * cfg.val_split))
    train_ds, val_ds = torch.utils.data.random_split(
        dataset, [len(dataset) - n_val, n_val],
        generator=torch.Generator().manual_seed(cfg.seed))
    train_loader = make_loader(train_ds, cfg, shuffle=True)
    val_loader   = make_loader(val_ds,   cfg, shuffle=False)
    print(f"  Clips this run - train: {len(train_ds)}  val: {len(val_ds)}")

    train_labels  = [dataset.samples[i][2] for i in train_ds.indices]
    class_weights = compute_class_weights(train_labels, cfg.num_event_classes, device)
    wt_str = "  ".join(f"{EVENT_NAMES[i]}:{class_weights[i].item():.2f}"
                       for i in range(cfg.num_event_classes))
    print(f"  Class weights: {wt_str}")

    model     = build_temporal_model(cfg).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=cfg.event_lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=cfg.event_epochs * TOTAL_RUNS)
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
    scaler    = GradScaler(device="cuda", enabled=cfg.mixed_precision)

    # Resume from last checkpoint if one exists
    model, optimizer, scheduler, start_epoch = resume_checkpoint(
        model, optimizer, scheduler, "event_detector", cfg.ckpt_dir, device)

    # Each run trains cfg.event_epochs epochs on top of previous runs
    end_epoch = start_epoch + cfg.event_epochs - 1
    print(f"  Training epochs {start_epoch} -> {end_epoch}")

    best_val   = float("inf")
    epoch_pbar = tqdm(range(start_epoch, end_epoch + 1), desc="Phase 1 epochs")
    for epoch in epoch_pbar:
        t0         = time.time()
        train_loss = train_one_epoch_event(
            model, train_loader, optimizer, criterion, scaler, cfg, device)
        scheduler.step()

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for feat, flow, y in val_loader:
                feat = feat.to(device)
                y    = y.to(device)
                flow = flow.to(device) if cfg.use_optical_flow else None
                with autocast(device_type=device.type, enabled=cfg.mixed_precision):
                    val_loss += criterion(model(feat, flow), y).item()
        val_loss /= max(len(val_loader), 1)

        elapsed = time.time() - t0
        notes   = f"run{RUN_ID}"
        if val_loss < best_val:
            best_val = val_loss
            save_checkpoint(model, "event_detector", epoch, cfg.ckpt_dir)
            notes += " checkpoint saved"

        training_logger.log("phase1_event", epoch, end_epoch,
                            train_loss, val_loss, best_val, elapsed, notes)
        epoch_pbar.set_postfix(train=f"{train_loss:.4f}",
                               val=f"{val_loss:.4f}",
                               time=f"{elapsed:.1f}s")

    print(f"\n  OK Phase 1 Run {RUN_ID} complete. Best val loss: {best_val:.4f}")
    return model


event_model = phase1_event_detection(matches, cfg, device)


  PHASE 1 - Event Detection  [Run 4/4]
  Loading CachedEventDataset...


  Matches:   0%|          | 0/25 [00:00<?, ?it/s]

  CachedEventDataset: 3567 clips
    Shot      : 508
    Foul      : 1304
    Goal      : 94
    Corner    : 233
    Offside   : 109
    Kickoff   : 139
    Other     : 1180
  Clips this run - train: 3032  val: 535
  Class weights: Shot:0.68  Foul:0.42  Goal:1.54  Corner:1.00  Offside:1.54  Kickoff:1.29  Other:0.44
  Resumed: event_detector_epoch039.pt  (continuing from epoch 40)
  Training epochs 40 -> 79


/home/sysadm/miniconda3/envs/torch_env/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:182: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(


Phase 1 epochs:   0%|          | 0/40 [00:00<?, ?it/s]

...

  batches:   0%|          | 0/189 [00:00<?, ?it/s]

  batches:   0%|          | 0/189 [00:00<?, ?it/s]

  batches:   0%|          | 0/189 [00:00<?, ?it/s]

  batches:   0%|          | 0/189 [00:00<?, ?it/s]


  OK Phase 1 Run 4 complete. Best val loss: 1.3835


## Cell 12 - Phase 2: Tactical GNN
Trains on tracking `gt.txt` files if available, otherwise falls back to synthetic data.

In [13]:
def phase2_tactical_gnn(matches, cfg, device):
    print("=" * 55)
    print("  PHASE 2 - Tactical GNN")
    print("=" * 55)

    TRACKING_DIR = str(Path(cfg.dataset_dir).parent / "tracking")
    try:
        dataset = TrajectoryDataset(
            tracking_dir=TRACKING_DIR,
            split="train",
            num_players=cfg.num_players,
            stride=5,
        )
    except FileNotFoundError as e:
        print(f"  WARNING: {e}")
        print("  Falling back to synthetic data.")
        dataset = _SyntheticTrajectoryDataset(len(matches) * 200, cfg.num_players)

    n_val = max(1, int(len(dataset) * cfg.val_split))
    train_ds, val_ds = torch.utils.data.random_split(
        dataset, [len(dataset) - n_val, n_val])

    def gnn_loader(ds):
        return DataLoader(ds, batch_size=1, shuffle=True, num_workers=0)

    train_loader = gnn_loader(train_ds)
    val_loader   = gnn_loader(val_ds)
    print(f"  Samples - train: {len(train_ds)}  val: {len(val_ds)}")

    model     = TacticalGNN(in_feat=4, hidden=cfg.gnn_hidden,
                            num_classes=cfg.num_formations).to(device)
    optimizer = optim.Adam(model.parameters(), lr=cfg.tactical_lr)
    criterion = nn.CrossEntropyLoss()

    best_val   = float("inf")
    epoch_pbar = tqdm(range(1, cfg.tactical_epochs + 1), desc="Phase 2 epochs")
    for epoch in epoch_pbar:
        t0 = time.time()
        model.train()
        total_loss = 0.0
        for nf, ei, label in train_loader:
            nf    = nf.squeeze(0).to(device)
            ei    = ei.squeeze(0).to(device)
            label = label.to(device)
            optimizer.zero_grad()
            loss  = criterion(model(nf, ei).unsqueeze(0), label)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        train_loss = total_loss / max(len(train_loader), 1)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for nf, ei, label in val_loader:
                nf    = nf.squeeze(0).to(device)
                ei    = ei.squeeze(0).to(device)
                label = label.to(device)
                val_loss += criterion(model(nf, ei).unsqueeze(0), label).item()
        val_loss /= max(len(val_loader), 1)

        elapsed = time.time() - t0
        notes   = ""
        if val_loss < best_val:
            best_val = val_loss
            save_checkpoint(model, "tactical_gnn", epoch, cfg.ckpt_dir)
            notes = "checkpoint saved"

        training_logger.log("phase2_tactical", epoch, cfg.tactical_epochs,
                            train_loss, val_loss, best_val, elapsed, notes)
        epoch_pbar.set_postfix(train=f"{train_loss:.4f}",
                               val=f"{val_loss:.4f}",
                               time=f"{elapsed:.1f}s")

    print(f"\n  OK Phase 2 complete. Best val loss: {best_val:.4f}")
    return model


tactical_model = phase2_tactical_gnn(matches, cfg, device)


  PHASE 2 - Tactical GNN
  Falling back to synthetic data.
  Samples - train: 4250  val: 750


Phase 2 epochs:   0%|          | 0/15 [00:00<?, ?it/s]

15:57:44  INFO        Saved: tactical_gnn_epoch001.pt



  OK Phase 2 complete. Best val loss: 2.2985


## Cell 13 - Phase 3: Tension Regression
Trains on this run's match chunk. **Automatically resumes from the last saved checkpoint.**

In [14]:
def phase3_tension_regression(matches, cfg, device):
    print("=" * 55)
    print(f"  PHASE 3 - Tension Regression  [Run {RUN_ID}/{TOTAL_RUNS}]")
    print("=" * 55)

    dataset = CachedTensionDataset(matches, cfg)

    if len(dataset) == 0:
        raise RuntimeError(
            "CachedTensionDataset is empty.\n"
            "Run feature_extraction.ipynb first.")

    n_val = max(1, int(len(dataset) * cfg.val_split))
    train_ds, val_ds = torch.utils.data.random_split(
        dataset, [len(dataset) - n_val, n_val],
        generator=torch.Generator().manual_seed(cfg.seed))
    train_loader = make_loader(train_ds, cfg)
    val_loader   = make_loader(val_ds, cfg, shuffle=False)
    print(f"  Clips this run - train: {len(train_ds)}  val: {len(val_ds)}")

    model     = TensionRegressor(cfg.feature_dim).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=cfg.tension_lr)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=cfg.tension_epochs * TOTAL_RUNS)
    criterion = nn.MSELoss()
    scaler    = GradScaler(device="cuda", enabled=cfg.mixed_precision)

    # Resume from last checkpoint if one exists
    model, optimizer, scheduler, start_epoch = resume_checkpoint(
        model, optimizer, scheduler, "tension_regressor", cfg.ckpt_dir, device)

    end_epoch = start_epoch + cfg.tension_epochs - 1
    print(f"  Training epochs {start_epoch} -> {end_epoch}")

    best_val   = float("inf")
    epoch_pbar = tqdm(range(start_epoch, end_epoch + 1), desc="Phase 3 epochs")
    for epoch in epoch_pbar:
        t0 = time.time()
        model.train()
        total_loss = 0.0
        optimizer.zero_grad()
        for step, (x, y) in enumerate(train_loader):
            x, y = x.to(device), y.to(device).float()
            with autocast(device_type=device.type, enabled=cfg.mixed_precision):
                loss = criterion(model(x), y) / cfg.grad_accum_steps
            scaler.scale(loss).backward()
            if (step + 1) % cfg.grad_accum_steps == 0:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
            total_loss += loss.item() * cfg.grad_accum_steps
        train_loss = total_loss / max(len(train_loader), 1)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device).float()
                with autocast(device_type=device.type, enabled=cfg.mixed_precision):
                    val_loss += criterion(model(x), y).item()
        val_loss /= max(len(val_loader), 1)

        elapsed = time.time() - t0
        notes   = f"run{RUN_ID}"
        if val_loss < best_val:
            best_val = val_loss
            save_checkpoint(model, "tension_regressor", epoch, cfg.ckpt_dir)
            notes += " checkpoint saved"

        training_logger.log("phase3_tension", epoch, end_epoch,
                            train_loss, val_loss, best_val, elapsed, notes)
        epoch_pbar.set_postfix(train=f"{train_loss:.4f}",
                               val=f"{val_loss:.4f}",
                               time=f"{elapsed:.1f}s")

    print(f"\n  OK Phase 3 Run {RUN_ID} complete. Best val loss: {best_val:.4f}")
    return model


tension_model = phase3_tension_regression(matches, cfg, device)


  PHASE 3 - Tension Regression  [Run 4/4]
  Loading CachedTensionDataset...


  Matches:   0%|          | 0/25 [00:00<?, ?it/s]

  CachedTensionDataset: 4774 clips  (avg tension: 0.360)
  Clips this run - train: 4058  val: 716
  Resumed: tension_regressor_epoch029.pt  (continuing from epoch 30)
  Training epochs 30 -> 39


Phase 3 epochs:   0%|          | 0/10 [00:00<?, ?it/s]

16:00:25  INFO        Saved: tension_regressor_epoch030.pt
16:00:25  INFO        Saved: tension_regressor_epoch031.pt
16:00:27  INFO        Saved: tension_regressor_epoch034.pt
16:00:28  INFO        Saved: tension_regressor_epoch035.pt
16:00:30  INFO        Saved: tension_regressor_epoch039.pt



  OK Phase 3 Run 4 complete. Best val loss: 0.0642


## Cell 14 - Phase 4: Language Model (GPT-2)

In [15]:
def phase4_language_model(cfg, device):
    print("=" * 55)
    print("  PHASE 4 - Language Model (GPT-2)")
    print("=" * 55)

    if GPT2LMHeadModel is None:
        print("  transformers not installed - skipping Phase 4.")
        return None, None

    tokenizer = GPT2Tokenizer.from_pretrained(cfg.lm_model_name)
    tokenizer.pad_token = tokenizer.eos_token
    lm   = GPT2LMHeadModel.from_pretrained(cfg.lm_model_name).to(device)

    state_dim = cfg.num_event_classes + cfg.num_formations + 1
    lm_embed  = lm.config.n_embd
    head      = CommentaryHead(state_dim, lm_embed).to(device)

    params    = list(lm.parameters()) + list(head.parameters())
    optimizer = optim.AdamW(params, lr=cfg.lm_lr, weight_decay=0.01)
    scaler    = GradScaler(device="cuda", enabled=cfg.mixed_precision)

    # Structured training data: (text, event_idx, tension)
    sample_data = [
        ("Shot from outside the box, deflected wide for a corner.", 0, 0.70),
        ("The striker fires at goal - saved by the keeper!",         0, 0.80),
        ("Foul given on the edge of the box. Dangerous free kick.",  1, 0.60),
        ("Yellow card shown for the reckless challenge in midfield.",1, 0.55),
        ("Goal! Brilliant finish curled into the top corner.",       2, 1.00),
        ("Goal! The keeper had no chance with that strike.",         2, 1.00),
        ("Corner kick swings in - headed clear by the defence.",     3, 0.50),
        ("Short corner, one-two and a shot blocked behind.",         3, 0.50),
        ("Flag goes up for offside. The attack breaks down.",        4, 0.40),
        ("Offside trap works perfectly, no goal.",                   4, 0.35),
        ("Kick-off as the second half gets underway.",               5, 0.30),
        ("Ball played back safely, keeper collects.",                6, 0.10),
        ("Throw-in deep in midfield, slow build up play.",           6, 0.15),
    ] * 80

    texts = [d[0] for d in sample_data]
    states_list = []
    for _, ev_idx, tension in sample_data:
        s = torch.zeros(state_dim)
        s[ev_idx] = 1.0
        s[-1]     = tension
        states_list.append(s)
    all_states = torch.stack(states_list)

    enc = tokenizer(texts, return_tensors="pt", padding=True,
                    truncation=True, max_length=cfg.max_commentary_len)
    input_ids = enc["input_ids"]
    attn_mask = enc["attention_mask"]

    lm.train(); head.train()
    epoch_pbar = tqdm(range(1, cfg.lm_epochs + 1), desc="Phase 4 epochs")
    for epoch in epoch_pbar:
        t0 = time.time(); total_loss = 0.0
        perm = torch.randperm(len(input_ids))
        n_batches = 0; optimizer.zero_grad()

        for i in range(0, len(input_ids) - cfg.batch_size, cfg.batch_size):
            idx   = perm[i: i + cfg.batch_size]
            ids   = input_ids[idx].to(device)
            mask  = attn_mask[idx].to(device)
            state = all_states[idx].to(device)

            with autocast(device_type=device.type, enabled=cfg.mixed_precision):
                prefix       = head(state).unsqueeze(1)
                embeds       = lm.transformer.wte(ids)
                inputs       = torch.cat([prefix, embeds[:, :-1]], dim=1)
                labels       = ids.clone(); labels[:, 0] = -100
                prefix_mask  = torch.ones(mask.shape[0], 1,
                                          device=device, dtype=mask.dtype)
                full_mask    = torch.cat([prefix_mask, mask[:, :-1]], dim=1)
                out  = lm(inputs_embeds=inputs,
                          attention_mask=full_mask, labels=labels)
                loss = out.loss / cfg.grad_accum_steps

            scaler.scale(loss).backward()
            if (i // cfg.batch_size + 1) % cfg.grad_accum_steps == 0:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(params, 1.0)
                scaler.step(optimizer); scaler.update(); optimizer.zero_grad()

            total_loss += loss.item() * cfg.grad_accum_steps
            n_batches  += 1

        avg_loss = total_loss / max(n_batches, 1)
        elapsed  = time.time() - t0
        save_checkpoint(lm,   "language_model",  epoch, cfg.ckpt_dir)
        save_checkpoint(head, "commentary_head", epoch, cfg.ckpt_dir)
        training_logger.log("phase4_lm", epoch, cfg.lm_epochs,
                            avg_loss, epoch_time=elapsed, notes="lm+head saved")
        epoch_pbar.set_postfix(loss=f"{avg_loss:.4f}", time=f"{elapsed:.1f}s")

    print("\n  OK Phase 4 complete.")
    return lm, head


lm, lm_head = phase4_language_model(cfg, device)


  PHASE 4 - Language Model (GPT-2)


16:00:31  INFO      HTTP Request: HEAD https://huggingface.co/gpt2/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
16:00:31  INFO      HTTP Request: GET https://huggingface.co/api/models/gpt2/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
16:00:32  INFO      HTTP Request: GET https://huggingface.co/api/models/openai-community/gpt2/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
16:00:32  INFO      HTTP Request: GET https://huggingface.co/api/models/gpt2/tree/main?recursive=true&expand=false "HTTP/1.1 307 Temporary Redirect"
16:00:32  INFO      HTTP Request: GET https://huggingface.co/api/models/openai-community/gpt2/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
16:00:33  INFO      HTTP Request: HEAD https://huggingface.co/gpt2/resolve/main/config.json "HTTP/1.1 200 OK"
16:00:33  INFO      HTTP Request: HEAD https://huggingface.co/gpt2/resolve/main/model.safetensors "HTTP/1.1 302

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

16:00:37  INFO      HTTP Request: HEAD https://huggingface.co/gpt2/resolve/main/generation_config.json "HTTP/1.1 200 OK"


Phase 4 epochs:   0%|          | 0/5 [00:00<?, ?it/s]

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.
16:00:47  INFO        Saved: language_model_epoch001.pt
16:00:47  INFO        Saved: commentary_head_epoch001.pt
16:00:54  INFO        Saved: language_model_epoch002.pt
16:00:54  INFO        Saved: commentary_head_epoch002.pt
16:01:02  INFO        Saved: language_model_epoch003.pt
16:01:02  INFO        Saved: commentary_head_epoch003.pt
16:01:09  INFO        Saved: language_model_epoch004.pt
16:01:09  INFO        Saved: commentary_head_epoch004.pt
16:01:17  INFO        Saved: language_model_epoch005.pt
16:01:17  INFO        Saved: commentary_head_epoch005.pt



  OK Phase 4 complete.


## Cell 15 - Summary & Training Log
Shows checkpoints saved so far and the full training log across all runs.

In [16]:
print("=" * 55)
print(f"  Run {RUN_ID}/{TOTAL_RUNS} complete")
print("=" * 55)
ckpt_files = sorted(Path(cfg.ckpt_dir).glob("*.pt"))
if ckpt_files:
    for f in ckpt_files:
        size_mb = f.stat().st_size / 1e6
        print(f"  {f.name:<45}  {size_mb:5.1f} MB")
else:
    print("  No checkpoints found.")

print(f"\n  Checkpoint directory : {cfg.ckpt_dir}")
print(f"  Training log         : {cfg.log_file}")

if RUN_ID < TOTAL_RUNS:
    print(f"\n  NEXT STEP: Set RUN_ID = {RUN_ID + 1} in Cell 3 and re-run the notebook.")
else:
    print(f"\n  ALL {TOTAL_RUNS} RUNS COMPLETE. Training finished across all matches.")

training_logger.summary()
training_logger.tail(20)


  Run 4/4 complete
  commentary_head_epoch001.pt                      0.8 MB
  commentary_head_epoch002.pt                      0.8 MB
  commentary_head_epoch003.pt                      0.8 MB
  commentary_head_epoch004.pt                      0.8 MB
  commentary_head_epoch005.pt                      0.8 MB
  event_detector_epoch001.pt                      14.8 MB
  event_detector_epoch002.pt                      14.8 MB
  event_detector_epoch003.pt                      14.8 MB
  event_detector_epoch004.pt                      14.8 MB
  event_detector_epoch005.pt                      14.8 MB
  event_detector_epoch006.pt                      14.8 MB
  event_detector_epoch007.pt                      14.8 MB
  event_detector_epoch008.pt                      14.8 MB
  event_detector_epoch009.pt                      14.8 MB
  event_detector_epoch010.pt                      14.8 MB
  event_detector_epoch011.pt                      14.8 MB
  event_detector_epoch012.pt                      14.